# 🛒 Machine Learning Escalable con Apache Spark MLlib
## Análisis de Caso – Predicción de Compras en E-commerce
---
**Objetivo:** Construir un modelo de clasificación supervisada que prediga si un cliente realizará una compra en los próximos días, usando Apache Spark MLlib para procesamiento distribuido y escalable.

**Flujo de trabajo:**

| Paso | Acción |
|------|--------|
| 1 | Importar librerías y crear sesión Spark |
| 2 | Cargar y explorar los datos |
| 3 | Seleccionar y limpiar variables |
| 4 | Vectorizar características (VectorAssembler + StandardScaler) |
| 5 | Dividir en Train / Test |
| 6 | Entrenar Regresión Logística |
| 7 | Entrenar Random Forest |
| 8 | Ajustar hiperparámetros con CrossValidator |
| 9 | Evaluar modelos (AUC-ROC, Accuracy, F1) |
| 10 | Informe y recomendaciones |


---
## Paso 1 – Importar librerías y crear la sesión Spark

### ¿Por qué Spark?
Los sistemas de ML tradicionales (scikit-learn, pandas) procesan datos **en una sola máquina**. Cuando el volumen supera los gigabytes, la memoria se agota.  
Apache Spark **distribuye** el procesamiento entre múltiples nodos, permitiendo manejar millones de registros con el **mismo código**.

### Componentes importados
| Módulo | Para qué sirve |
|--------|---------------|
| `SparkSession` | Punto de entrada de toda aplicación Spark |
| `VectorAssembler` | Combina múltiples columnas en un único vector de features |
| `StandardScaler` | Normaliza las features (media=0, std=1) |
| `LogisticRegression` | Modelo de clasificación lineal |
| `RandomForestClassifier` | Ensemble de árboles de decisión |
| `CrossValidator` | Búsqueda de hiperparámetros con validación cruzada |
| `BinaryClassificationEvaluator` | Calcula AUC-ROC |


In [73]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, count
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

# Creamos la sesión Spark
# local[*] significa: usa todos los núcleos disponibles en esta máquina
spark = SparkSession.builder \
    .appName("EcommercePurchasePrediction") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Sesión Spark creada | Versión: {spark.version}")


Sesión Spark creada | Versión: 4.1.1


---
## Paso 2 – Cargar y explorar los datos

### Descripción del dataset maestro
El archivo `dataset_maestro.csv` integra tres fuentes de datos:

| Fuente | Variables |
|--------|-----------|
| **Perfil** | edad, género, región, antigüedad, membresía |
| **Navegación** | páginas vistas, tiempo en sesión, visitas, ítems en carrito |
| **Historial** | compras 30d/90d, ticket promedio, devoluciones |

La variable objetivo es **`comprara`** (1 = sí comprará, 0 = no comprará).

### ¿Qué es `inferSchema=True`?
Spark detecta automáticamente el tipo de cada columna (entero, decimal, texto).  
Sin este parámetro, todas las columnas serían texto (*string*).


In [74]:
# Cargamos el dataset maestro
df = spark.read.csv("dataset_maestro.csv", header=True, inferSchema=True)

print(f"Dimensiones: {df.count()} filas × {len(df.columns)} columnas")
print()

# Vista rápida del esquema
df.printSchema()


Dimensiones: 1000 filas × 24 columnas

root
 |-- cliente_id: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- genero: string (nullable = true)
 |-- region: string (nullable = true)
 |-- antiguedad_dias: integer (nullable = true)
 |-- nivel_membresia: string (nullable = true)
 |-- paginas_vistas_7d: integer (nullable = true)
 |-- tiempo_sesion_min: double (nullable = true)
 |-- visitas_7d: integer (nullable = true)
 |-- busquedas_7d: integer (nullable = true)
 |-- items_carrito: integer (nullable = true)
 |-- abandono_carrito: integer (nullable = true)
 |-- categoria_favorita: string (nullable = true)
 |-- compras_30d: integer (nullable = true)
 |-- compras_90d: integer (nullable = true)
 |-- ticket_promedio: double (nullable = true)
 |-- devolucion_30d: integer (nullable = true)
 |-- calificacion_promedio: double (nullable = true)
 |-- descuento_usado: integer (nullable = true)
 |-- comprara: integer (nullable = true)
 |-- genero_cod: integer (nullable = true)
 |-- m

### ⚠️ Nota sobre `isnan()` en Spark

`isnan()` **solo funciona con columnas numéricas de punto flotante** (`double`, `float`).  
Si se aplica a una columna de texto (`string`) o entero (`int`), Spark lanza un error de tipo.

Por eso usamos una función auxiliar `check_null()` que revisa el tipo de cada columna antes de decidir cómo verificar los nulos:

```python
# double/float → puede haber NaN o NULL
isnan(col(c)) | col(c).isNull()

# int/string   → solo puede haber NULL
col(c).isNull()
```


In [75]:
# Verificar valores nulos por columna
# ─────────────────────────────────────────────────────────────────
# ⚠️  PROBLEMA COMÚN: isnan() en Spark solo funciona con columnas
#     numéricas de punto flotante (double, float).
#     Si la aplicamos a un string (genero, region…) Spark lanza error.
#
# SOLUCIÓN: detectar el tipo de cada columna con df.dtypes y
#           aplicar la verificación correcta según corresponda:
#   → double / float  : isnan(col) | col.isNull()   (NaN + NULL)
#   → int / string    : col.isNull()                (solo NULL)
# ─────────────────────────────────────────────────────────────────

def check_null(col_name, dtype):
    """Cuenta nulos de forma segura según el tipo de columna."""
    if dtype in ("double", "float"):
        return count(when(isnan(col(col_name)) | col(col_name).isNull(), col_name)).alias(col_name)
    else:
        return count(when(col(col_name).isNull(), col_name)).alias(col_name)

print("Valores nulos por columna:")
df.select([
    check_null(c, t) for c, t in df.dtypes
]).show(vertical=True)


Valores nulos por columna:
-RECORD 0--------------------
 cliente_id            | 0   
 edad                  | 0   
 genero                | 0   
 region                | 0   
 antiguedad_dias       | 0   
 nivel_membresia       | 0   
 paginas_vistas_7d     | 0   
 tiempo_sesion_min     | 0   
 visitas_7d            | 0   
 busquedas_7d          | 0   
 items_carrito         | 0   
 abandono_carrito      | 0   
 categoria_favorita    | 0   
 compras_30d           | 0   
 compras_90d           | 0   
 ticket_promedio       | 0   
 devolucion_30d        | 0   
 calificacion_promedio | 0   
 descuento_usado       | 0   
 comprara              | 0   
 genero_cod            | 0   
 membresia_cod         | 0   
 region_cod            | 0   
 categoria_cod         | 0   



In [76]:
# Distribución de la variable objetivo
print("Distribución de 'comprara':")
df.groupBy("comprara").count().orderBy("comprara").show()

# Con porcentaje
total = df.count()
df.groupBy("comprara").count().orderBy("comprara") \
  .withColumn("porcentaje", (col("count") / total * 100).cast("decimal(5,2)")) \
  .show()


Distribución de 'comprara':
+--------+-----+
|comprara|count|
+--------+-----+
|       0|  420|
|       1|  580|
+--------+-----+

+--------+-----+----------+
|comprara|count|porcentaje|
+--------+-----+----------+
|       0|  420|     42.00|
|       1|  580|     58.00|
+--------+-----+----------+



In [77]:
# Estadísticas descriptivas de variables numéricas clave
print("Estadísticas descriptivas:")
df.select(
    "edad", "antiguedad_dias", "paginas_vistas_7d",
    "visitas_7d", "items_carrito", "compras_30d", "ticket_promedio"
).describe().show()


Estadísticas descriptivas:
+-------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|summary|              edad|  antiguedad_dias|paginas_vistas_7d|       visitas_7d|    items_carrito|      compras_30d|  ticket_promedio|
+-------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|  count|              1000|             1000|             1000|             1000|             1000|             1000|             1000|
|   mean|            40.986|          932.411|           59.661|           14.503|            7.295|            3.563| 87.2005199999999|
| stddev|13.497851532595707|534.4340706343996|34.98453939952613|8.654530136979812|4.439333585425033|2.273147201172462|80.81521011366189|
|    min|                18|                1|                0|                0|                0|                0|            10.08|
|    max|     

---
## Paso 3 – Selección y limpieza de variables

### ¿Por qué seleccionar variables?
No todas las columnas son útiles para el modelo:
- Las columnas de texto original (`genero`, `region`, etc.) ya tienen su versión codificada numéricamente (`genero_cod`, `region_cod`, etc.).
- Incluir columnas redundantes puede generar **multicolinealidad** y ruido.

### Variables seleccionadas
Elegimos 18 features agrupadas por categoría:

```
Perfil del cliente   → edad, antiguedad_dias, genero_cod, membresia_cod, region_cod
Comportamiento       → paginas_vistas_7d, tiempo_sesion_min, visitas_7d,
                       busquedas_7d, items_carrito, abandono_carrito, categoria_cod
Historial de compras → compras_30d, compras_90d, ticket_promedio,
                       devolucion_30d, calificacion_promedio, descuento_usado
```


In [78]:
# Variables predictoras (features)
FEATURES = [
    # Perfil del cliente
    "edad", "antiguedad_dias", "genero_cod", "membresia_cod", "region_cod",
    # Comportamiento de navegación
    "paginas_vistas_7d", "tiempo_sesion_min", "visitas_7d",
    "busquedas_7d", "items_carrito", "abandono_carrito", "categoria_cod",
    # Historial de compras
    "compras_30d", "compras_90d", "ticket_promedio",
    "devolucion_30d", "calificacion_promedio", "descuento_usado",
]

TARGET = "comprara"   # variable objetivo: 1=comprará, 0=no comprará

# Seleccionar columnas y eliminar filas con nulos
df_modelo = df.select(FEATURES + [TARGET]).dropna()

print(f"Features seleccionadas : {len(FEATURES)}")
print(f"   Filas tras dropna()   : {df_modelo.count()}")
df_modelo.show(5)


Features seleccionadas : 18
   Filas tras dropna()   : 1000
+----+---------------+----------+-------------+----------+-----------------+-----------------+----------+------------+-------------+----------------+-------------+-----------+-----------+---------------+--------------+---------------------+---------------+--------+
|edad|antiguedad_dias|genero_cod|membresia_cod|region_cod|paginas_vistas_7d|tiempo_sesion_min|visitas_7d|busquedas_7d|items_carrito|abandono_carrito|categoria_cod|compras_30d|compras_90d|ticket_promedio|devolucion_30d|calificacion_promedio|descuento_usado|comprara|
+----+---------------+----------+-------------+----------+-----------------+-----------------+----------+------------+-------------+----------------+-------------+-----------+-----------+---------------+--------------+---------------------+---------------+--------+
|  56|           1240|         0|            0|         4|               91|            19.43|        20|          38|            8|          

---
## Paso 4 – Vectorizar características con VectorAssembler y StandardScaler

### ¿Por qué vectorizar?
MLlib exige que **todas las features estén en una sola columna** de tipo `Vector`.  
`VectorAssembler` toma N columnas y las concatena en un vector denso por fila.

**Ejemplo mini:**
```
Antes:   edad=25  |  visitas_7d=10  |  items_carrito=3
Después: features = DenseVector([25.0, 10.0, 3.0])
```

### ¿Por qué normalizar?
`StandardScaler` transforma cada feature para que tenga **media = 0** y **desviación estándar = 1**:

$$x_{norm} = \frac{x - \mu}{\sigma}$$

**¿Por qué importa?** La Regresión Logística es sensible a la escala. Sin normalizar, una variable como `ticket_promedio` (valores entre 10 y 500) dominaría sobre `abandono_carrito` (0 o 1), distorsionando los coeficientes.


In [79]:
# VectorAssembler: une todas las features en un vector "features_raw"
assembler = VectorAssembler(inputCols=FEATURES, outputCol="features_raw")

# StandardScaler: normaliza el vector (media=0, std=1)
# - withMean=True  → resta la media (centra los datos)
# - withStd=True   → divide por la desviación estándar (escala los datos)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

print("VectorAssembler configurado → outputCol: 'features_raw'")
print("StandardScaler configurado  → outputCol: 'features'")

# Demostración: cómo queda el vector
demo = assembler.transform(df_modelo.limit(3))
demo.select("features_raw").show(3, truncate=False)


VectorAssembler configurado → outputCol: 'features_raw'
StandardScaler configurado  → outputCol: 'features'
+-------------------------------------------------------------------------------------+
|features_raw                                                                         |
+-------------------------------------------------------------------------------------+
|[56.0,1240.0,0.0,0.0,4.0,91.0,19.43,20.0,38.0,8.0,1.0,1.0,4.0,3.0,60.87,1.0,4.0,1.0] |
|[46.0,407.0,1.0,1.0,2.0,52.0,6.86,16.0,35.0,10.0,0.0,3.0,0.0,18.0,248.04,0.0,1.5,1.0]|
|[32.0,399.0,0.0,1.0,1.0,105.0,2.39,14.0,29.0,8.0,0.0,1.0,5.0,19.0,160.45,0.0,2.5,1.0]|
+-------------------------------------------------------------------------------------+



---
## Paso 5 – División Train / Test

### ¿Por qué dividir?
Si entrenamos el modelo con TODOS los datos y lo evaluamos con los mismos datos,
obtendremos métricas artificialmente altas (**sobreajuste**).  
Al separar un conjunto de prueba, evaluamos cómo el modelo generaliza a datos **que nunca vio**.

### Proporción usada: 80 % / 20 %
- **80 % Train**: el modelo aprende los patrones.
- **20 % Test**: evaluamos el rendimiento real.

El parámetro `seed=42` garantiza que la división sea siempre la misma
(reproducibilidad del experimento).


In [80]:
# Dividimos el dataset: 80% entrenamiento, 20% prueba
train_df, test_df = df_modelo.randomSplit([0.8, 0.2], seed=42)

print(f"Train : {train_df.count()} registros ({train_df.count()/df_modelo.count()*100:.1f}%)")
print(f"Test  : {test_df.count()} registros ({test_df.count()/df_modelo.count()*100:.1f}%)")

# Distribución del target en cada conjunto (sanity check)
print("\nDistribución de 'comprara' en Train:")
train_df.groupBy("comprara").count().orderBy("comprara").show()
print("Distribución de 'comprara' en Test:")
test_df.groupBy("comprara").count().orderBy("comprara").show()


Train : 838 registros (83.8%)
Test  : 162 registros (16.2%)

Distribución de 'comprara' en Train:
+--------+-----+
|comprara|count|
+--------+-----+
|       0|  360|
|       1|  478|
+--------+-----+

Distribución de 'comprara' en Test:
+--------+-----+
|comprara|count|
+--------+-----+
|       0|   60|
|       1|  102|
+--------+-----+



---
## Paso 6 – Modelo 1: Regresión Logística

### ¿Cómo funciona?
La Regresión Logística calcula la **probabilidad** de que un cliente compre:

$$P(\text{compra}=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots)}}$$

La función `1 / (1 + e^(-z))` se llama **sigmoide** y siempre produce valores entre 0 y 1.

**Regla de decisión:**
- Si P > 0.5 → predice **comprará** (clase 1)
- Si P ≤ 0.5 → predice **no comprará** (clase 0)

**Mini-ejemplo numérico:**
```
z = 0.5×(compras_30d=3) + 0.3×(items_carrito=5) − 0.8 = 1.7
P = 1 / (1 + e^{-1.7}) ≈ 0.85  → Comprará ✔
```

### Pipeline: concepto clave
Un `Pipeline` encadena transformaciones y el modelo en **un solo objeto**.  
Así, cuando hacemos `pipeline.fit(train)`, Spark aplica automáticamente:
`assembler → scaler → modelo`


In [81]:
# Definimos la Regresión Logística
# - regParam   : penalización para evitar sobreajuste (mayor = más regularización)
# - maxIter    : número máximo de iteraciones del optimizador
# - elasticNetParam: 0=Ridge (L2), 1=Lasso (L1)
lr = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET,
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0
)

# Pipeline: assembler → scaler → modelo LR
pipeline_lr = Pipeline(stages=[assembler, scaler, lr])

print("Entrenando Regresión Logística...")
model_lr = pipeline_lr.fit(train_df)
print("Modelo entrenado")

# Coeficientes del modelo
lr_model = model_lr.stages[-1]
print(f"\n   Intercepto (β₀): {lr_model.intercept:.4f}")
print(f"   Número de coeficientes: {len(lr_model.coefficients)}")


Entrenando Regresión Logística...
Modelo entrenado

   Intercepto (β₀): 0.3497
   Número de coeficientes: 18


In [82]:
# Ver predicciones sobre el conjunto de prueba
preds_lr = model_lr.transform(test_df)
print("Ejemplo de predicciones (Regresión Logística):")
preds_lr.select("comprara", "prediction", "probability").show(8, truncate=False)


Ejemplo de predicciones (Regresión Logística):
+--------+----------+----------------------------------------+
|comprara|prediction|probability                             |
+--------+----------+----------------------------------------+
|0       |1.0       |[0.49310344015966123,0.5068965598403388]|
|1       |1.0       |[0.4793799420420432,0.5206200579579567] |
|1       |1.0       |[0.272245137685534,0.7277548623144661]  |
|1       |1.0       |[0.29425641236223743,0.7057435876377626]|
|1       |1.0       |[0.11927740762267024,0.8807225923773297]|
|1       |1.0       |[0.31723300836783797,0.682766991632162] |
|1       |0.0       |[0.7224428586980927,0.27755714130190734]|
|1       |1.0       |[0.17071875977273876,0.8292812402272612]|
+--------+----------+----------------------------------------+
only showing top 8 rows


---
## Paso 7 – Modelo 2: Random Forest

### ¿Cómo funciona?
Random Forest construye **múltiples árboles de decisión** en paralelo.
Cada árbol se entrena con:
- Una **muestra aleatoria** de los datos (bagging)
- Un **subconjunto aleatorio** de las features

La predicción final se obtiene por **votación**: si 70 de 100 árboles predicen "comprará", la clase predicha es 1.

**Ventajas sobre Regresión Logística:**
- Captura relaciones **no lineales** (ej: clientes con muchas visitas Y carrito lleno compran mucho más)
- No requiere normalización
- Más robusto al sobreajuste

**Parámetros clave:**
- `numTrees=100` → 100 árboles en el ensemble
- `maxDepth=5`   → profundidad máxima de cada árbol (mayor = más complejo)


In [83]:
# Random Forest no necesita StandardScaler
# → pipeline solo con assembler
rf = RandomForestClassifier(
    featuresCol="features_raw",
    labelCol=TARGET,
    numTrees=100,
    maxDepth=5,
    seed=42
)

pipeline_rf = Pipeline(stages=[assembler, rf])

print("Entrenando Random Forest (100 árboles)...")
model_rf = pipeline_rf.fit(train_df)
print("Modelo Random Forest entrenado")


Entrenando Random Forest (100 árboles)...
Modelo Random Forest entrenado


In [84]:
# Importancia de variables (feature importance)
# Mide cuánto contribuye cada variable a las predicciones del modelo
rf_model = model_rf.stages[-1]
feature_imp = sorted(
    zip(FEATURES, rf_model.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)

print("\nImportancia de variables – Random Forest:")
print(f"  {'Variable':<25} {'Importancia':>10}  Gráfico")
print(f"  {'-'*25} {'-'*10}  {'─'*20}")
for feat, imp in feature_imp:
    bar = "█" * int(imp * 300)
    print(f"  {feat:<25} {imp:>10.4f}  {bar}")



Importancia de variables – Random Forest:
  Variable                  Importancia  Gráfico
  ------------------------- ----------  ────────────────────
  items_carrito                 0.3128  █████████████████████████████████████████████████████████████████████████████████████████████
  visitas_7d                    0.3015  ██████████████████████████████████████████████████████████████████████████████████████████
  paginas_vistas_7d             0.0529  ███████████████
  compras_90d                   0.0479  ██████████████
  calificacion_promedio         0.0410  ████████████
  edad                          0.0316  █████████
  antiguedad_dias               0.0310  █████████
  ticket_promedio               0.0286  ████████
  tiempo_sesion_min             0.0273  ████████
  busquedas_7d                  0.0249  ███████
  compras_30d                   0.0227  ██████
  membresia_cod                 0.0141  ████
  categoria_cod                 0.0133  ███
  devolucion_30d                0.01

---
## Paso 8 – Ajuste de hiperparámetros con CrossValidator

### ¿Qué es la validación cruzada (k-fold)?
En lugar de dividir los datos en Train/Test una sola vez, la validación cruzada
repite el proceso K veces con diferentes particiones:

```
Fold 1: [████████] [██] → entrena en 80%, evalúa en 20%
Fold 2: [████][████████] → entrena en diferente 80%, evalúa en otro 20%
Fold 3: [████████][████] → ídem
```
El resultado final es el **promedio** de las K evaluaciones. Esto da una estimación
más robusta del rendimiento real.

### Grid de hiperparámetros
Probamos todas las combinaciones de:

| `regParam` | `maxIter` |
|:---:|:---:|
| 0.01 | 50 |
| 0.01 | 100 |
| 0.1 | 50 |
| 0.1 | 100 |
| 0.5 | 50 |
| 0.5 | 100 |

Total: **6 combinaciones × 3 folds = 18 entrenamientos**  
El CrossValidator elige automáticamente la combinación con mejor AUC-ROC.


In [85]:
# LR fresca para la búsqueda de hiperparámetros
lr_cv = LogisticRegression(featuresCol="features", labelCol=TARGET)
pipeline_cv = Pipeline(stages=[assembler, scaler, lr_cv])

# Definimos el grid de búsqueda
param_grid = (
    ParamGridBuilder()
    .addGrid(lr_cv.regParam, [0.01, 0.1, 0.5])
    .addGrid(lr_cv.maxIter,  [50, 100])
    .build()
)

print(f"   Combinaciones a probar: {len(param_grid)}")

# Evaluador: usamos AUC-ROC como criterio de selección
evaluator_auc = BinaryClassificationEvaluator(
    labelCol=TARGET,
    metricName="areaUnderROC"
)

# CrossValidator: 3 folds
cv = CrossValidator(
    estimator=pipeline_cv,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_auc,
    numFolds=3,
    seed=42
)

print("Ejecutando CrossValidator (18 entrenamientos)... puede tomar unos minutos.")
cv_model = cv.fit(train_df)

# Mostrar los mejores hiperparámetros
best_lr = cv_model.bestModel.stages[-1]
print(f"\nMejores hiperparámetros encontrados:")
print(f"   regParam : {best_lr.getRegParam()}")
print(f"   maxIter  : {best_lr.getMaxIter()}")


   Combinaciones a probar: 6
Ejecutando CrossValidator (18 entrenamientos)... puede tomar unos minutos.

Mejores hiperparámetros encontrados:
   regParam : 0.1
   maxIter  : 50


In [86]:
# Ver el ranking de todas las combinaciones probadas
print("\nAUC-ROC promedio por combinación de hiperparámetros:")
print(f"  {'regParam':>10}  {'maxIter':>8}  {'AUC-ROC':>10}")
print(f"  {'-'*10}  {'-'*8}  {'-'*10}")
for params, avg_metric in zip(param_grid, cv_model.avgMetrics):
    rp = params[lr_cv.regParam]
    mi = params[lr_cv.maxIter]
    print(f"  {rp:>10}  {mi:>8}  {avg_metric:>10.4f}")



AUC-ROC promedio por combinación de hiperparámetros:
    regParam   maxIter     AUC-ROC
  ----------  --------  ----------
        0.01        50      0.8570
        0.01       100      0.8570
         0.1        50      0.8576
         0.1       100      0.8576
         0.5        50      0.8574
         0.5       100      0.8574


---
## Paso 9 – Evaluación de los modelos

### Métricas usadas

#### 1. AUC-ROC (Área bajo la curva ROC)
Mide qué tan bien el modelo **separa** clientes que comprarán de los que no.
- **1.0** = perfecto (nunca se equivoca)
- **0.5** = equivale a adivinar al azar

#### 2. Accuracy (Exactitud)
$$\text{Accuracy} = \frac{\text{Predicciones correctas}}{\text{Total de predicciones}}$$

**Limitación**: si el 90% de los clientes no compran, un modelo que siempre predice "no compra" tiene accuracy=90% pero es inútil.

#### 3. F1-Score
Combina **Precisión** (¿de los que predije que compran, cuántos realmente compran?)
y **Recall** (¿de los que realmente compran, cuántos detecté?):

$$F1 = 2 \times \frac{\text{Precisión} \times \text{Recall}}{\text{Precisión} + \text{Recall}}$$

Es la métrica más equilibrada cuando las clases están desbalanceadas.


In [87]:
# Configurar evaluadores
evaluator_auc = BinaryClassificationEvaluator(labelCol=TARGET, metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol=TARGET, metricName="accuracy")
evaluator_f1  = MulticlassClassificationEvaluator(labelCol=TARGET, metricName="f1")

def evaluar_modelo(modelo, nombre, test_data):
    """Evalúa un modelo sobre el conjunto de prueba y retorna métricas."""
    preds = modelo.transform(test_data)
    auc = evaluator_auc.evaluate(preds)
    acc = evaluator_acc.evaluate(preds)
    f1  = evaluator_f1.evaluate(preds)

    print(f"\n  ╔══════════════════════════════════════╗")
    print(f"  ║  Modelo: {nombre:<27} ║")             
    print(f"  ╠══════════════════════════════════════╣")
    print(f"  ║  AUC-ROC  : {auc:.4f}                   ║")
    print(f"  ║  Accuracy : {acc:.4f}  ({acc*100:.1f}%)          ║")
    print(f"  ║  F1-Score : {f1:.4f}                   ║")
    print(f"  ╚══════════════════════════════════════╝")
    return {"Modelo": nombre, "AUC-ROC": round(auc,4),
            "Accuracy": round(acc,4), "F1-Score": round(f1,4)}

resultados = []
resultados.append(evaluar_modelo(model_lr,  "Regresión Logística",  test_df))
resultados.append(evaluar_modelo(model_rf,  "Random Forest",        test_df))
resultados.append(evaluar_modelo(cv_model,  "LR + CrossValidation", test_df))



  ╔══════════════════════════════════════╗
  ║  Modelo: Regresión Logística         ║
  ╠══════════════════════════════════════╣
  ║  AUC-ROC  : 0.8042                   ║
  ║  Accuracy : 0.7284  (72.8%)          ║
  ║  F1-Score : 0.7301                   ║
  ╚══════════════════════════════════════╝

  ╔══════════════════════════════════════╗
  ║  Modelo: Random Forest               ║
  ╠══════════════════════════════════════╣
  ║  AUC-ROC  : 0.7806                   ║
  ║  Accuracy : 0.7037  (70.4%)          ║
  ║  F1-Score : 0.7047                   ║
  ╚══════════════════════════════════════╝

  ╔══════════════════════════════════════╗
  ║  Modelo: LR + CrossValidation        ║
  ╠══════════════════════════════════════╣
  ║  AUC-ROC  : 0.8042                   ║
  ║  Accuracy : 0.7284  (72.8%)          ║
  ║  F1-Score : 0.7301                   ║
  ╚══════════════════════════════════════╝


In [88]:
# Tabla comparativa de resultados
import pandas as pd
tabla = pd.DataFrame(resultados)
print("\nTABLA COMPARATIVA DE MODELOS")
print(tabla.to_string(index=False))

# Identificar el mejor modelo
mejor = tabla.loc[tabla["AUC-ROC"].idxmax()]
print(f"\nMejor modelo: {mejor['Modelo']}  (AUC-ROC = {mejor['AUC-ROC']})")



TABLA COMPARATIVA DE MODELOS
              Modelo  AUC-ROC  Accuracy  F1-Score
 Regresión Logística   0.8042    0.7284    0.7301
       Random Forest   0.7806    0.7037    0.7047
LR + CrossValidation   0.8042    0.7284    0.7301

Mejor modelo: Regresión Logística  (AUC-ROC = 0.8042)


---
## Paso 10 – Informe de resultados y recomendaciones

### Resumen del flujo de trabajo

```
CSV (datos crudos)
     ↓
VectorAssembler → DenseVector con 18 features
     ↓
StandardScaler  → features normalizadas (solo para LR)
     ↓
Train/Test split (80% / 20%, seed=42)
     ↓
┌─────────────────────┐    ┌──────────────────────────┐
│ Regresión Logística │    │      Random Forest       │
│  regParam=0.1       │    │  100 árboles, depth=5    │
│  maxIter=100        │    │                          │
└─────────────────────┘    └──────────────────────────┘
               ↓
     CrossValidator (LR)
     6 combinaciones × 3 folds
               ↓
   Evaluación: AUC-ROC · Accuracy · F1-Score
```

### Recomendaciones para mejorar el modelo

**1. Feature Engineering**
- `ratio_compras = compras_30d / (compras_90d + 1)` → tendencia de compra
- `engagement = visitas_7d × items_carrito` → interacción compuesta
- `días_desde_ultima_compra` → aporta mucho en e-commerce

**2. Probar otros algoritmos MLlib**
- `GBTClassifier` (Gradient Boosted Trees): suele superar a RF en datos tabulares
- `LinearSVC`: alternativa para clasificación binaria

**3. Manejo de desbalance**
Si el target es muy desbalanceado (ej: 95% no-compra):
- Usar `weightCol` para dar más peso a la clase minoritaria
- Aplicar SMOTE (librería `imblearn`) antes de convertir a Spark

**4. Infraestructura en producción**
- Reemplazar `local[*]` por un clúster Databricks, Amazon EMR o GCP Dataproc
- El mismo código funciona sin cambios: solo cambia el `master()`
- Re-entrenar mensualmente para capturar cambios de comportamiento


In [89]:
# Cerramos la sesión Spark al terminar
# Siempre es buena práctica liberar los recursos
spark.stop()
print("Sesión Spark cerrada. Notebook completado exitosamente.")


Sesión Spark cerrada. Notebook completado exitosamente.
